# 🧹 Lab 11: 리소스 정리 — 안전 인벤토리 & 단계별 Teardown

워크숍이 끝난 뒤 **무엇이 배포되어 있는지 먼저 확인**하고, **랩이 만든 아티팩트만
선별 삭제**하거나 **리소스 그룹 전체를 삭제**합니다.

> ⚠️ **안전장치** — 삭제 셀은 기본적으로 `DRY_RUN = True` 라서 **실제로 지우지 않고 계획만 출력**합니다.
> 내용을 확인한 뒤 `DRY_RUN = False`(+ 전체삭제는 `CONFIRM = "DELETE"`)로 바꿔 실행하세요.
> 인벤토리·검증 셀은 모두 **읽기 전용**이라 안전합니다.

| 단계 | 셀 | 파괴성 |
|---|---|---|
| Phase 1 | 리소스 인벤토리 | 읽기 전용 ✅ |
| Phase 2 | 랩 아티팩트 선별 정리(제품/구독/백엔드/NV/API/Workbook) | `DRY_RUN` 가드 |
| Phase 3 | 리소스 그룹 전체 삭제 | `DRY_RUN` + `CONFIRM` 가드 |
| Phase 4 | 삭제 확인 + soft-delete purge 안내 | 읽기 전용 ✅ |

In [ ]:
# ─── 환경 설정 + Azure 헬퍼 ───
import os, json, subprocess, tempfile, re
from dotenv import load_dotenv
load_dotenv("../../.env", override=True)

def az(args):
    r = subprocess.run(["az"] + args, capture_output=True, text=True)
    return r.stdout.strip(), r.stderr.strip(), r.returncode
def az_json(args):
    out, err, rc = az(args)
    if rc != 0 or not out: return None
    try: return json.loads(out)
    except json.JSONDecodeError: return out

SUBSCRIPTION_ID = os.getenv("AZURE_SUBSCRIPTION_ID") or (az_json(["account","show","--query","id","-o","json"]) or "")
RESOURCE_GROUP  = os.getenv("RESOURCE_GROUP","")
APIM_NAME       = os.getenv("APIM_NAME","")
if not (RESOURCE_GROUP and APIM_NAME):
    apims = az_json(["apim","list","--query","[].{name:name,rg:resourceGroup}","-o","json"]) or []
    if apims:
        APIM_NAME = APIM_NAME or apims[0]["name"]; RESOURCE_GROUP = RESOURCE_GROUP or apims[0]["rg"]
assert SUBSCRIPTION_ID and RESOURCE_GROUP, "❌ az login / RESOURCE_GROUP 를 확인하세요."
ARM_API = "2024-06-01-preview"; WB_API = "2023-06-01"
RG_BASE   = f"https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}"
ARM_BASE  = f"{RG_BASE}/providers/Microsoft.ApiManagement/service/{APIM_NAME}" if APIM_NAME else ""

def rest(method, url, body=None, query=None):
    args = ["rest","--method",method,"--url",url]
    if query: args += ["--query",query,"-o","json"]
    tmp = None
    if body is not None:
        tmp = tempfile.NamedTemporaryFile("w",suffix=".json",delete=False)
        json.dump(body,tmp); tmp.close()
        args += ["--headers","Content-Type=application/json","--body",f"@{tmp.name}"]
    out, err, rc = az(args)
    if tmp: os.unlink(tmp.name)
    return out, err, rc
def arm_get(path, api=ARM_API, base=None):
    base = base if base is not None else ARM_BASE
    out, err, rc = rest("GET", f"{base}{path}?api-version={api}", query="value")
    if rc != 0 or not out: return []
    try: return json.loads(out) or []
    except json.JSONDecodeError: return []

print("✅ 설정 완료")
print(f"   구독: {SUBSCRIPTION_ID}")
print(f"   RG  : {RESOURCE_GROUP}")
print(f"   APIM: {APIM_NAME or '(없음)'}")

---
## Phase 1 — 리소스 인벤토리 (읽기 전용)

리소스 그룹 전체와 APIM 내부(제품·구독·백엔드·Named Value·API)·Workbook 을 조회하고,
**기반 리소스**(base)와 **랩이 만든 아티팩트**(lab)를 구분해 표시합니다.

In [ ]:
# ─── Phase 1: 인벤토리 ───
# 기반(삭제 금지) 식별 규칙
BASE_PRODUCTS = {"starter", "unlimited"}
BASE_SUB_NAMES = {"master"}
KEEP_BACKEND = re.compile(r"^(aoai-|openai-backend-pool|gemini-backend)")
KEEP_NV      = re.compile(r"^gemini-api-key")
BASE_APIS    = {"azure-openai", "echo-api", "gemini"}
# 랩 아티팩트 식별 규칙(선별 삭제 후보)
LAB_BACKEND  = re.compile(r"(openai-compat|-direct-|^anthropic-backend$)")
LAB_NV       = {"openai-api-key", "anthropic-api-key"}
LAB_API      = re.compile(r"multicloud")
LAB_WB       = re.compile(r"(거버넌스|AI Gateway)")

def tag(is_lab): return "🧪 lab" if is_lab else "🔒 base"

# 1) RG 리소스 타입별 요약
res = az_json(["resource","list","-g",RESOURCE_GROUP,"--query","[].{t:type,n:name}","-o","json"]) or []
from collections import Counter
by_type = Counter(r["t"] for r in res)
print(f"■ 리소스 그룹 '{RESOURCE_GROUP}' — 총 {len(res)}개 리소스")
for t,n in sorted(by_type.items()): print(f"   {n:>2} × {t}")

plan = {"products":[], "subscriptions":[], "backends":[], "namedValues":[], "apis":[], "workbooks":[]}

if ARM_BASE:
    print("\n■ APIM 제품(Products)")
    for p in arm_get("/products"):
        name = p["name"]; lab = name not in BASE_PRODUCTS
        if lab: plan["products"].append(name)
        print(f"   {tag(lab)}  {name}  ({p.get('properties',{}).get('displayName','')})")

    print("\n■ APIM 구독(Subscriptions)")
    for s in arm_get("/subscriptions"):
        name = s["name"]; scope = s.get("properties",{}).get("scope","")
        base = (name in BASE_SUB_NAMES or scope.endswith("/products/starter")
                or scope.endswith("/products/unlimited"))
        lab = not base
        if lab: plan["subscriptions"].append(name)
        disp = s.get("properties",{}).get("displayName") or "(no name)"
        print(f"   {tag(lab)}  {name}  ({disp})")

    print("\n■ APIM 백엔드(Backends)")
    for b in arm_get("/backends"):
        name = b["name"]; lab = bool(LAB_BACKEND.search(name)) or not KEEP_BACKEND.search(name)
        if lab and LAB_BACKEND.search(name): plan["backends"].append(name)
        print(f"   {tag(lab and bool(LAB_BACKEND.search(name)))}  {name}")

    print("\n■ APIM Named Values")
    for v in arm_get("/namedValues"):
        name = v["name"]; lab = name in LAB_NV
        if lab: plan["namedValues"].append(name)
        print(f"   {tag(lab)}  {name}")

    print("\n■ APIM API")
    for a in arm_get("/apis"):
        name = a["name"]; lab = bool(LAB_API.search(name)) or name not in BASE_APIS
        flag = bool(LAB_API.search(name))
        if flag: plan["apis"].append(name)
        print(f"   {tag(flag)}  {name}")

print("\n■ Azure Monitor Workbook")
wb_out, _, wb_rc = rest("GET", f"{RG_BASE}/providers/Microsoft.Insights/workbooks?api-version={WB_API}&category=workbook", query="value")
wbs = json.loads(wb_out) if (wb_rc == 0 and wb_out) else []
for w in wbs:
    disp = w.get("properties",{}).get("displayName",""); lab = bool(LAB_WB.search(disp))
    if lab: plan["workbooks"].append(w["name"])
    print(f"   {tag(lab)}  {w['name']}  ({disp})")

print("\n" + "="*60)
print("선별 삭제 후보(🧪 lab):")
for k,v in plan.items(): print(f"   {k:<14}: {len(v)}개  {v if v else ''}")
globals()["CLEANUP_PLAN"] = plan

---
## Phase 2 — 랩 아티팩트 선별 정리 (`DRY_RUN` 가드)

Phase 1 이 찾은 **🧪 lab 아티팩트만** 삭제합니다(기반 리소스·`gemini-api-key-*`·기본 제품/구독은 보존).
랩 노트북이 중간에 중단돼 잔여물이 남았을 때 유용합니다.

> 기본 `DRY_RUN = True` → 삭제 계획만 출력. 실제 삭제하려면 `DRY_RUN = False` 로 변경.

In [ ]:
# ─── Phase 2: 랩 아티팩트 선별 정리 ───
DRY_RUN = True   # ← 실제 삭제하려면 False

plan = globals().get("CLEANUP_PLAN")
assert plan is not None, "먼저 Phase 1 인벤토리 셀을 실행하세요."

# 삭제 순서: 구독 → 제품 → API → 백엔드 → NamedValue → Workbook
targets = []
for s in plan["subscriptions"]: targets.append(("subscription", f"{ARM_BASE}/subscriptions/{s}?api-version={ARM_API}", s))
for p in plan["products"]:      targets.append(("product",      f"{ARM_BASE}/products/{p}?api-version={ARM_API}&deleteSubscriptions=true", p))
for a in plan["apis"]:          targets.append(("api",          f"{ARM_BASE}/apis/{a}?api-version={ARM_API}", a))
for b in plan["backends"]:      targets.append(("backend",      f"{ARM_BASE}/backends/{b}?api-version={ARM_API}", b))
for v in plan["namedValues"]:   targets.append(("namedValue",   f"{ARM_BASE}/namedValues/{v}?api-version={ARM_API}", v))
for w in plan["workbooks"]:     targets.append(("workbook",     f"{RG_BASE}/providers/Microsoft.Insights/workbooks/{w}?api-version={WB_API}", w))

if not targets:
    print("✅ 정리할 랩 아티팩트가 없습니다. 환경이 깨끗합니다.")
else:
    print(f"{'[DRY-RUN] 삭제 예정' if DRY_RUN else '삭제 실행'} — {len(targets)}개\n")
    for kind, url, name in targets:
        if DRY_RUN:
            print(f"   [계획] {kind:<12} {name}")
        else:
            out, err, rc = rest("DELETE", url)
            print(f"   {'🧹 삭제' if rc==0 else '⚠️ 실패'} {kind:<12} {name}" + ("" if rc==0 else f"  ({err[:80]})"))
    if DRY_RUN:
        print("\n➡️  확인 후 DRY_RUN = False 로 바꿔 다시 실행하세요.")

---
## Phase 3 — 리소스 그룹 전체 삭제 (`DRY_RUN` + `CONFIRM` 가드)

APIM·Azure OpenAI·App Insights 를 포함한 **리소스 그룹 전체**를 삭제합니다(가장 파괴적).
시간당 과금되는 APIM(Developer ~$50/월)을 확실히 정리하는 방법입니다.

> 이중 안전장치: `DRY_RUN = False` **그리고** `CONFIRM = "DELETE"` 둘 다여야 실제 삭제됩니다.

In [ ]:
# ─── Phase 3: 리소스 그룹 전체 삭제 ───
DRY_RUN = True      # ← False 로 변경
CONFIRM = ""        # ← 정확히 "DELETE" 입력해야 실행

cmd = ["group","delete","--name",RESOURCE_GROUP,"--yes","--no-wait"]
print(f"대상 리소스 그룹: {RESOURCE_GROUP}")
print(f"실행될 명령: az {' '.join(cmd)}\n")
if DRY_RUN or CONFIRM != "DELETE":
    print("[DRY-RUN] 실제 삭제하지 않았습니다.")
    print("  → 삭제하려면: DRY_RUN = False  그리고  CONFIRM = \"DELETE\" 로 설정 후 재실행")
else:
    out, err, rc = az(cmd)
    if rc == 0:
        print("🧹 리소스 그룹 삭제 요청 완료(백그라운드 진행). Phase 4 로 확인하세요.")
    else:
        print(f"⚠️ 실패: {err[:200]}")

---
## Phase 4 — 삭제 확인 + soft-delete purge (읽기 전용)

리소스 그룹 삭제 진행 상태를 확인하고, Azure OpenAI/APIM 의 **soft-delete** 잔여를 조회합니다.
soft-delete 가 남으면 같은 이름 재배포 시 충돌하므로 purge 명령을 안내합니다.

In [ ]:
# ─── Phase 4: 삭제 확인 + soft-delete ───
state = az_json(["group","show","--name",RESOURCE_GROUP,"--query","properties.provisioningState","-o","json"])
if state is None:
    print(f"✅ 리소스 그룹 '{RESOURCE_GROUP}' 이(가) 없습니다(삭제 완료 또는 미존재).")
else:
    print(f"리소스 그룹 상태: {state}  (삭제 중이면 'Deleting')")

print("\n■ Cognitive Services(OpenAI) soft-delete 목록")
deleted = az_json(["cognitiveservices","account","list-deleted",
                   "--query","[].{name:name,location:location}","-o","json"]) or []
if deleted:
    for d in deleted:
        print(f"   - {d['name']} ({d['location']})")
    print("   purge: az cognitiveservices account purge --name <이름> "
          f"--resource-group {RESOURCE_GROUP} --location <region>")
else:
    print("   (없음)")

print("\n■ APIM soft-delete 목록")
apim_del = az_json(["apim","deletedservice","list",
                    "--query","[].{name:name,location:location}","-o","json"]) or []
if apim_del:
    for d in apim_del:
        print(f"   - {d['name']} ({d['location']})")
    print("   purge: az apim deletedservice purge --service-name <이름> --location <region>")
else:
    print("   (없음)")

print("\n체크리스트")
print("   [ ] az group show 시 ResourceGroupNotFound")
print("   [ ] 포털에서 리소스 그룹 미표시")
print("   [ ] (선택) soft-delete purge 완료")

---
### ✅ 정리 완료

- **Phase 2** 는 워크숍 중 잔여 랩 아티팩트만 안전하게 제거합니다(반복 실습 시 유용).
- **Phase 3** 는 전체 teardown 으로 과금을 확실히 중단합니다.
- 이 노트북은 [README](./README.md) 의 CLI 절차를 자동화·가드한 버전입니다.

→ [메인 README로 돌아가기](../../README.md)